# RetainIQ · EDA Walkthrough

> **Synthetic data (70,000 rows, 37 columns).** Generated by `src/data_generation/generate_synthetic_data.py`, modeled on a real (confidential, ~2k-row) hotel group project.  
> Run `python run_pipeline.py` for the full analysis. This notebook is an interactive walkthrough of the data before modelling.

**Analytical sequence followed in this project:**
1. EDA (this notebook) → understand distributions and raw relationships
2. Logistic regression → identify significant predictors, directions, odds ratios
3. ML benchmark → 9 classifiers on the significant feature set
4. Revenue regression, forecasting, segmentation, revenue-management analyses

In [ ]:
import sys; from pathlib import Path
ROOT = Path('..').resolve(); sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style='whitegrid', palette='muted')
from src.ingestion.load_data import load
df = pd.read_csv(ROOT / 'data' / 'synthetic' / 'median_inn_synthetic.csv')
df['Check_In_Date'] = pd.to_datetime(df['Check_In_Date'])
print(f'Shape: {df.shape}  |  Columns: {list(df.columns)}')
df.head()

## 1. Dataset overview and data types

In [ ]:
print('=== dtypes ===')
print(df.dtypes)
print('\n=== Missing values ===')
missing = df.isnull().sum()
print(missing[missing > 0])
print('\n=== Numeric summary ===')
df.describe().T

## 2. Cancellation rate — overall and by booking source
This is the variable of interest. We look at raw rates before any modelling.

In [ ]:
print(f'Overall cancellation rate: {df["Is_Cancelled"].mean():.2%}')
print()
cr = df.groupby('Booking_Source')['Is_Cancelled'].mean().sort_values(ascending=False)
print('Cancellation rate by source:')
print(cr.round(3))
cr.plot(kind='barh', color='#E85D04', title='Cancellation rate by booking source')
plt.xlabel('Cancellation rate'); plt.tight_layout(); plt.show()

## 3. Lead time — the strongest single predictor
Already visible in raw data before any modelling.

In [ ]:
df['lead_bucket'] = pd.cut(df['Lead_Time_Days'], bins=[0,7,14,30,60,999],
    labels=['0-7d','8-14d','15-30d','31-60d','60d+'])
lt = df.groupby('lead_bucket', observed=True)['Is_Cancelled'].mean()
print('Cancellation rate by lead time:')
print(lt.round(3))
lt.plot(kind='bar', color=['#3fb950','#4aa8ff','#d29922','#E85D04','#cf222e'],
        title='Cancellation rises sharply with booking lead time', rot=0)
plt.ylabel('Cancellation rate'); plt.tight_layout(); plt.show()

## 4. Guest history signals
Previous cancellations and successful stays — key behavioural features.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df.groupby('Previous_Cancellations')['Is_Cancelled'].mean().plot(
    kind='bar', ax=axes[0], color='#E85D04',
    title='Cancel rate by prior cancellations', rot=0)
axes[0].set_ylabel('Cancellation rate')
df.groupby('Previous_Successful_Bookings')['Is_Cancelled'].mean().plot(
    kind='bar', ax=axes[1], color='#3fb950',
    title='Cancel rate by prior successful stays', rot=0)
axes[1].set_ylabel('Cancellation rate')
plt.tight_layout(); plt.show()
print('Prior cancellations vs cancel rate:')
print(df.groupby('Previous_Cancellations')['Is_Cancelled'].mean().round(3))

## 5. Special requests — commitment signal

In [ ]:
sr = df.groupby('N_Special_Requests')['Is_Cancelled'].mean()
print('Cancellation rate by number of special requests:')
print(sr.round(3))
sr.plot(kind='bar', color='#1A1A2E', title='More special requests → less likely to cancel', rot=0)
plt.ylabel('Cancellation rate'); plt.tight_layout(); plt.show()

## 6. Monthly revenue — seasonality

In [ ]:
hon = df[df.Is_Cancelled == 0]
rev = hon.assign(m=hon['Check_In_Date'].dt.to_period('M').astype(str)).groupby('m')['Total_Amount'].sum()
rev.plot(figsize=(12,4), marker='o', ms=3, color='#E85D04',
         title='Monthly revenue — April/September peaks visible')
plt.ylabel('Revenue (₹)'); plt.xticks(rotation=75, fontsize=8)
plt.tight_layout(); plt.show()

## 7. Payment mode effect

In [ ]:
pm = df.groupby('Payment_Mode')['Is_Cancelled'].mean()
print('Cancellation rate by payment mode:')
print(pm.round(3))
print()
print('Insight: Prepaid OTA bookings cancel MORE than Pay-at-Hotel.')
print('Prepaid OTA guests can cancel at zero personal cost (OTA absorbs the refund).')
print('This is confirmed by the logistic regression — OR 0.35x for Pay-at-Hotel vs Prepaid.')

## 8. Revenue concentration (Pareto)

In [ ]:
g = df.groupby('Primary_Guest_Name')['Total_Amount'].sum().sort_values(ascending=False)
cum = g.cumsum() / g.sum()
x = (cum.index.get_indexer(cum.index) + 1) / len(cum)
pct_18 = int(len(cum) * 0.18)
print(f'Top 18% of guests = {cum.iloc[pct_18]:.1%} of revenue')
plt.figure(figsize=(8, 5))
plt.plot(np.linspace(0, 1, len(cum)), cum.values, color='#E85D04', lw=2)
plt.axvline(0.18, color='#888', ls='--', lw=1)
plt.axhline(cum.iloc[pct_18], color='#888', ls='--', lw=1)
plt.title('Revenue concentration (Pareto)')
plt.xlabel('Cumulative share of guests'); plt.ylabel('Cumulative share of revenue')
plt.tight_layout(); plt.show()

## 9. Correlation heatmap — numeric features vs Is_Cancelled
Shows raw correlations before logistic regression — useful to spot obvious collinearity.

In [ ]:
num_df = df[['Lead_Time_Days','N_Adults','N_Children','N_Special_Requests',
             'Previous_Cancellations','Previous_Successful_Bookings',
             'Avg_Price_Per_Room','Rate_Premium_Pct','Nights_Booked','Is_Cancelled']]
corr = num_df.corr()
print('Correlation with Is_Cancelled (sorted):')
print(corr['Is_Cancelled'].drop('Is_Cancelled').sort_values().round(3))
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, linewidths=0.5)
plt.title('Numeric feature correlation matrix')
plt.tight_layout(); plt.show()

## Next steps
The EDA shows clear signals from `Lead_Time_Days`, `Previous_Cancellations`, `Booking_Source`, and `Payment_Mode`.  
The next step is **logistic regression** (`src/analysis/logistic_regression_primary.py`) to quantify each effect with proper statistical inference (p-values, odds ratios, confidence intervals) before building any ML models.  
Run `python run_pipeline.py` to execute the full analysis in order.